# 02 — Cognitive Memory Lab

> **AEE Bootcamp | Mini Project 03 — Part 2 [30 Marks]**  
> Tests all three memory tiers: Semantic (profiles), Long-Term RAG (Qdrant), and Short-Term (session buffer).

---

## The Problem This Solves

The core failure of a naïve chatbot is **statelessness**. When a Kapruka customer says *"I need the same cake as last year"*, a simple bot replies: *"I'm sorry, I don't have access to last year's order. What kind of cake was it?"* — forcing the customer to repeat themselves and eroding trust.

The **TEAM Agent** (Tiered, Enriched, Agentic Memory) solves this by combining three distinct memory types:

```
           ▲  Short-Term (ST)
          ▲▲  ─── Session-based conversation buffer (live context)
         ▲▲▲▲  LT-RAG
        ▲▲▲▲▲  ─── Qdrant vector store (crawled catalog.json)
       ▲▲▲▲▲▲▲  Semantic
      ▲▲▲▲▲▲▲▲  ─── JSON Recipient Profiles (preferences & allergies)
```

| Tier | Type | Storage | Purpose |
|---|---|---|---|
| **ST** | Short-Term | In-memory ring buffer | Keeps the last N conversation turns in the LLM prompt |
| **LT-RAG** | Long-Term | Qdrant vector DB | Semantic product search over the full catalog |
| **Semantic** | Long-Term | `profiles.json` (JSON) | Persistent per-recipient facts (likes, dislikes, allergies) |

---

> ⚠️ **Submission Rule**: No LangGraph/CrewAI. All memory modules (`session_buffer`, `vector_db`) must be hand-rolled.

## Cell 1 — Environment Setup & Module Imports

Adds the project root to `sys.path` so that the local `memory/` and `utils/` packages are importable. Three custom modules are loaded:

- `memory.session_buffer` — The Tier 3 (ST) sliding window conversation buffer.
- `memory.vector_db` — Qdrant client wrapper for Tier 2 (LT-RAG) ingestion and retrieval.
- `utils.config` — Reads `config/params.yaml` for environment-specific settings (Qdrant URL, score thresholds, etc.).

In [ ]:
import os
import sys
import json
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from memory.session_buffer import SessionBuffer
from memory.vector_db import ingest_catalog, retrieve_products
from utils.config import get_config

print("✅ Environment setup complete. Custom modules loaded successfully.")

## Cell 2 — Tier 1: Semantic Memory (User Profiles)

**Storage**: `data/profiles.json`  
**Access Pattern**: Key-value lookup by `customer_id` → `recipient_name`

Semantic memory stores **long-term, structured facts** about the people a customer buys gifts for. Unlike RAG, this data is not retrieved by similarity — it is fetched deterministically and injected directly into every catalog and reflection prompt.

**Example profile structure:**
```json
{
  "CUS_001": {
    "recipients": {
      "Wife": {
        "likes": ["dark chocolate", "orchids"],
        "dislikes": ["synthetic fragrances"],
        "allergies": ["peanuts", "tree nuts"],
        "past_orders": ["Nut-Free Chocolate Cake - 2023"]
      }
    }
  }
}
```

The `allergies` list is the critical field — it is the input to the **Reflection Loop** (Part 4) that prevents unsafe gift recommendations from reaching the customer.

In [ ]:
profiles_path = PROJECT_ROOT / "data" / "profiles.json"

if not profiles_path.exists():
    print(f"❌ Could not find {profiles_path}")
else:
    with open(profiles_path, "r", encoding="utf-8") as f:
        profiles_db = json.load(f)

    customer_id = "CUS_001"
    recipient   = "Wife"

    wife_profile = profiles_db.get(customer_id, {}).get("recipients", {}).get(recipient, {})

    print(f"👤 Profile loaded for {customer_id}'s {recipient}:")
    print(json.dumps(wife_profile, indent=4))
    print("\n✅ Tier 1: Semantic Memory is working!")

## Cell 3 — Tier 2: Long-Term Memory (Qdrant Vector DB Ingestion)

**Storage**: Qdrant collection `kapruka_products`  
**Input**: `data/catalog.json` (produced by Notebook 01)

This cell checks whether the Qdrant vector store has already been populated. Ingesting ~10,000 products involves:
1. Reading each product from `catalog.json`.
2. Calling an embedding model to convert the product description into a dense vector.
3. Upserting the vector + metadata payload into the `kapruka_products` collection.

> ⏱️ **First-time ingestion** can take several hours for a 10,000-item catalog due to embedding API rate limits. The existence check ensures subsequent notebook runs skip this expensive step.

**Qdrant payload schema per product:**
```
{ id, name, price, category, product_url, description (with SAFETY: tag) }
```

In [ ]:
from memory.vector_db import qdrant_client, ingest_catalog

print("🚀 Checking Qdrant Vector Database...")

if qdrant_client.collection_exists("kapruka_products"):
    print("✅ Collection 'kapruka_products' already exists!")
    print("⏭️ Skipping the 10,000 item ingestion to save time and API limits.")
else:
    print("⚠️ Collection not found. Starting ingestion (This may take hours for 10k items!)...")
    try:
        ingest_catalog()
    except Exception as e:
        print(f"❌ Ingestion failed: {e}")

## Cell 4 — Tier 2: Long-Term Memory (RAG Retrieval Test)

With the catalog ingested, this cell validates the **retrieval** side of the RAG pipeline.

**How retrieval works:**
1. The user's query (e.g. *"A nice chocolate cake for a birthday"*) is embedded using the same model as ingestion.
2. Qdrant performs an approximate nearest-neighbour search against the `kapruka_products` collection.
3. The top-`k` results (by cosine similarity) are returned as Python dicts.
4. The Catalog Agent uses these results to ground its gift recommendations in real, available products.

**Tuning tip**: If no results are returned, lower the `rag.score_threshold` in `config/params.yaml`. The default is conservative to avoid irrelevant recommendations.

In [ ]:
test_query = "A nice chocolate cake for a birthday"
print(f"🔎 Querying Database for: '{test_query}'\n")

try:
    results = retrieve_products(test_query, limit=3)

    if results:
        print(f"🎯 Found {len(results)} matches:")
        print("-" * 50)
        for idx, item in enumerate(results, 1):
            print(f"{idx}. {item.get('name')} | {item.get('price')}")
            print(f"   Category: {item.get('category')}")
        print("-" * 50)
        print("\n✅ Tier 2: Long-Term RAG Memory is working!")
    else:
        print("⚠️ No results found. You may need to lower the 'rag.score_threshold' in your config/params.yaml.")
except Exception as e:
    print(f"❌ Retrieval failed: {e}")

## Cell 5 — Tier 3: Short-Term Memory (Session Buffer)

**Implementation**: `memory/session_buffer.py` — `SessionBuffer` class  
**Mechanism**: A fixed-size ring buffer of (role, content) message pairs.

The `SessionBuffer` maintains the **live conversation context** injected into every LLM prompt. Its key design constraint is the `max_pairs` parameter — once the buffer is full, the oldest exchange is silently dropped. This prevents the prompt from growing unboundedly while keeping recent context.

**Capacity in this demo**: `max_pairs=3` → at most 3 user/assistant exchange pairs (6 messages) are retained.

The `get_history_string()` method formats the buffer as a plain-text dialogue string, ready to be inserted directly into a system or user prompt:
```
User: Hi, I need a gift for my wife.
Assistant: Ayubowan! I'd love to help. What does she like?
...
```

Note how the 4th pair (about nut allergies) correctly evicts the 1st pair when `max_pairs=3`.

In [ ]:
session = SessionBuffer(max_pairs=3)

session.add_message("user",      "Hi, I need a gift for my wife.")
session.add_message("assistant", "Ayubowan! I'd love to help. What does she like?")
session.add_message("user",      "She really loves chocolate, but she is allergic to nuts.")
session.add_message("assistant", "Noted! I will look for nut-free chocolate options.")
session.add_message("user",      "Do you have any cakes like that?")

print("📜 Formatted Conversation History for the Prompt:")
print("=" * 60)
print(session.get_history_string())
print("=" * 60)
print("\n✅ Tier 3: Short-Term Memory is working!")

---

## ✅ Part 2 Complete

All three memory tiers are validated and operational:

| Tier | Status |
|---|---|
| Semantic Memory (Recipient Profiles) | ✅ Loaded and queryable |
| Long-Term RAG (Qdrant) | ✅ Ingested and retrieving |
| Short-Term Buffer (Session) | ✅ Ring buffer working correctly |

**Next step → [03_specialist_orchestration_04_the_reflection_loop.ipynb](03_specialist_orchestration_04_the_reflection_loop.ipynb)**  
Wire all three memory tiers into the specialist agent routing pipeline and add the safety reflection loop.